# Regular expressions: precision & recall
Week 3 · Web and Text Analytics

All reviews are **real**, from the *Amazon Unlocked Mobile* dataset (Kaggle).

| Part | Task | Who writes the regex |
|---|---|---|
| **1. Worked example** | Find every review that mentions the **SIM card** | We do, step by step |
| **2. Your turn** | Find the **storage size** (16GB, 64GB, …) in each review | You do |
| **Extra practice** | Extract **dates** written in different formats: [Regex Dates · Interactive guide](https://kalampokis.github.io/Week3/RegexDates.html) | You do |

In every part we judge a regex the same way: compare what it finds against a **ground truth** written by a human, and report **precision** and **recall**.

Adapted from Jurafsky & Martin, *Speech and Language Processing* (3rd ed. draft), Ch. 2.

Run the cells from top to bottom (Shift+Enter).

In [ ]:
import re
import pandas as pd
from collections import Counter

---
# Part 1 · Worked example: SIM card
Ten reviews. Task: find every review that mentions the **SIM card**. We write three patterns, one after the other, and measure each one.

In [ ]:
# 10 reviews copied verbatim from Amazon_Unlocked_Mobile.csv (Reviews column)
sim_reviews = [
    "Great phone. Did not look used hardly at all. Worked very well. We just took the sim card from the old Verizon Samsung and put it in this one. Very seamless!",
    "Came exactly as described. Brand new iPhone 5s, unlocked. My husband put his old iPhones SIM card in it and it now works with AT&T",
    "I did not get a Sim card with phone",
    "Although the phone itself was good, it turns out to be a non-verizon machine. which doesn't work with verizon simcard. I think they didn't test it. before selling.",
    "My first iphone and i really like it. The phone works great and it came with sims card, charger, and headphones.",
    "Apple IPhone 5S is working perfectly. No problem getting phone connected thru Verizon service and their free SIMM Card. Very happy with Amazon timely service also.Thank you.",
    "Never going back to Android or Windows phone. Super reliable and simple to use. Battery life could be better, however.",
    "Phone purchased as replacement for similar phone. Phone was as described and arrived on time. Just like brand new.",
    "Simple to use. Easy set up. Amazing!",
    "It was simply awesome. No complaints at all.",
]

## The ground truth
A human read every review and decided: does it mention the SIM card? Without these labels, precision and recall cannot be measured.

In [ ]:
#           1     2     3     4     5     6     7      8      9      10
GT = [True, True, True, True, True, True, False, False, False, False]

## Evaluation function
For every review: did the regex match, and should it have?
- **TP**: matched and should have
- **FP**: matched but should not have
- **FN**: did not match but should have

$$\text{precision} = \frac{TP}{TP+FP} \qquad \text{recall} = \frac{TP}{TP+FN}$$

In [ ]:
def evaluate(pattern, texts, GT):
    tp = fp = fn = 0
    for i, (t, g) in enumerate(zip(texts, GT), start=1):
        found = re.search(pattern, t) is not None
        if found and g:
            tp += 1
        elif found and not g:
            fp += 1
            print(f"  FP  #{i}: {t[:60]}...")
        elif g:
            fn += 1
            print(f"  FN  #{i}: {t[:60]}...")
    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    print(f"TP={tp}  FP={fp}  FN={fn}   precision={precision:.0%}  recall={recall:.0%}")
    return precision, recall

## v1: the obvious first try

In [ ]:
_ = evaluate(r"sim", sim_reviews, GT)

Two kinds of error: it **misses** `SIM card` and `Sim card` (false negatives) and **finds** `simple`, `similar`, `simply` (false positives).

## v2: fix the false negatives (any case)

In [ ]:
_ = evaluate(r"[sS][iI][mM]", sim_reviews, GT)

Recall is now 100%, but precision barely moved: we also caught `Simple`.

## v3: fix the false positives (word boundaries)

In [ ]:
_ = evaluate(r"\b[sS][iI][mM][sS]?\b", sim_reviews, GT)

Precision is 100%, but recall fell to 67%: `simcard` and `SIMM Card` are written without a space or with a typo.

**Fixing one kind of error created the other.** Increasing recall means minimizing false negatives; increasing precision means minimizing false positives. The two efforts pull against each other.


---
# Part 2 · Your turn: storage size
Your product manager writes:

> *"Find out which storage size (16GB, 64GB, ...) each customer mentions in their review, so we can see which phones make people happy."*

You have **30 real reviews**. Write **one regex** that finds the storage size.

In [ ]:
#@title ▶ Run me first: 30 reviews + helper functions
phones = pd.DataFrame([
    (5, 'Apple iPhone 5s Verizon Wireless, 16GB, Space Gray'),
    (5, 'The phone was just as described: brand new factory GSM unlocked iPhone 5s 16gb!'),
    (5, 'I purchased the gold 16 GB variation of the iPhone 5s and it is so beautiful for a device.'),
    (5, 'I bought the iPhone 5s (16GB) in Space Gray from Express Mobile'),
    (5, 'I purchased a refurbished iPhone 5s, 32gb. The phone is as good as new.'),
    (5, 'This is a great Iphone 5S 64GB Factory Unlocked, had no problem getting it to fire up on Verizon'),
    (5, 'I still have about 38 GB of free space! I am EXTREMELY pleased!'),
    (5, 'PowerVR GT7600 six-core GPU, 2GB RAM, 1624 mAh battery.'),
    (5, '...the one I purchased fully supports AT&T 4G LTE.'),
    (5, 'Expect 3G during international use because this phone likely will not be capable of LTE'),
    (5, '...they offer the 16GB 5s at the full retail price of $649.99 (plus tax, of course).'),
    (5, 'Great condition'),
    (4, 'As of November 2015 this phones memory is just about full. 16 GB is simply NOT enough space.'),
    (5, 'I was fortunate to purchase a like-new 64gb iPhone 5s for 265USD in absolutely pristine condition'),
    (1, 'Verizon indicates that the phone is not compatible with their system'),
    (5, "It has 64 gigs, runs fast, is way more powerful than all my colleague's samsungs."),
    (5, 'Got my 32GB Gold iPhone from cell cow, phone works great!'),
    (5, 'Love my new iphone! expensive to pay for a 1g plan at virizen ($50).'),
    (5, "I'm not sure how I bought this phone for $350 with 64 gigabytes of internal memory but I did."),
    (5, '...the 64GB of storage is amazing compared to the 8GB of storage I had on my iPhone 4S.'),
    (5, 'No scratches dings or dents! Plus is 32g! my kid is loving it!'),
    (5, 'I have owned every iPhone since the 3G.'),
    (5, 'Absolutely the best phone I ever had ! 64 gb gives me all the storage I need !'),
    (1, "I don't even know if I got the 32 gig for that matter."),
    (5, 'I absolutely loved the 5G way beyond my expectations for my 1st iPhone'),
    (5, 'Unlocked, no scratches, and 32gbs. The only thing was it came with a cover and no charger'),
    (3, 'I recieved a iPhone 5 32GB instead of getting the iPhone 5S 64GB.'),
    (5, '...at the moment only data cards have access to the 4G network'),
    (5, 'no problems hooking up with att. great price for 64 g phone'),
    (1, 'i did not receive 32GB.. we paid fpr 32.. but received a 16,, not Happy at all'),
], columns=["rating", "review"])
phones.index += 1

def show(pattern, first=30):
    """Print the reviews with every match in [[double brackets]]."""
    for i, text in phones["review"].head(first).items():
        print(f"{i:>2}. {re.sub(pattern, lambda m: '[[' + m.group() + ']]', text) if pattern else text}")

def found(pattern):
    """List everything your regex found."""
    if not pattern:
        return " Write your regex in PATTERN first."
    return [m.group() for text in phones["review"] for m in re.finditer(pattern, text)]

def table(pattern):
    """Average rating per storage size (first size in each review)."""
    if not pattern:
        return " Write your regex in PATTERN first."
    sizes = [re.search(r"\d+", m.group()).group() + " GB" if (m := re.search(pattern, t)) else None
             for t in phones["review"]]
    return phones.assign(storage=sizes).groupby("storage")["rating"].agg(["count", "mean"]).round(2)

print("Ready: 30 reviews loaded.")

## Step 1 · Read the data
Copy reviews 1–15 into **[regex101.com](https://regex101.com)** (*TEST STRING*, flavor **Python**).
Which numbers are storage? Which only *look* like storage?

In [ ]:
show("", first=15)

**Your rules** (double-click to edit). These are your ground truth, written in words, just like `GT` in Part 1.
- Find: ...
- Do not find: ...

## Step 2 · Write the regex
Build your regex in regex101, then paste it below. You may ask an LLM, **but give it your rules**.

In [ ]:
PATTERN = r""

show(PATTERN, first=15)

## Step 3 · New data
Reviews 16–30 are new to you. Does your regex still work? Read the list: is every value a real storage size? Fix one thing in Step 2 and run again.

In [ ]:
show(PATTERN)
print(found(PATTERN))

## Step 4 · Use it (10')

In [ ]:
table(PATTERN)